# Comparação: avaliação humana (professores) × juízes LLM

1. Exporte a aba **respostas** do Google Sheet como `dados-professores/respostas_professores.csv`
2. Execute as células abaixo para comparar Likert 1–3 nas cinco dimensões de conteúdo (Tabela 1 do heatmap)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon

ROOT = Path("..")
CAMINHO_RESPOSTAS = ROOT / "dados-professores" / "respostas_professores.csv"
CAMINHO_REF_IA = ROOT / "dados-professores" / "referencia_ia.csv"
CAMINHO_MANIFEST = ROOT / "dados-professores" / "amostra_manifest.csv"
CAMINHO_CONSOLIDADO = (
    ROOT / "resultado_rubrica" / "prova-tecnologia-results-1500-rubrica-16dim-consolidado.csv"
)

DIMENSOES_TABELA1 = [
    "alinhamento_objetivos",
    "especificidade",
    "tom_motivacional",
    "pontos_fortes",
    "pontos_fracos",
]

COL_PROF = {
    "alinhamento_objetivos": "likert_alinhamento",
    "especificidade": "likert_especificidade",
    "tom_motivacional": "likert_tom",
    "pontos_fortes": "likert_fortes",
    "pontos_fracos": "likert_fracos",
}

ROTULOS_DIMENSAO = {
    "alinhamento_objetivos": "Alinhamento",
    "especificidade": "Especificidade",
    "tom_motivacional": "Tom motivacional",
    "pontos_fortes": "Pontos fortes",
    "pontos_fracos": "Pontos fracos",
}

ROTULO_JUIZ = {
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gpt-5-mini": "GPT-5-mini",
}

In [ ]:
if not CAMINHO_RESPOSTAS.exists():
    raise FileNotFoundError(
        f"Exporte a planilha para {CAMINHO_RESPOSTAS} (veja dados-professores/README.md)"
    )

respostas = pd.read_csv(CAMINHO_RESPOSTAS)
manifest = pd.read_csv(CAMINHO_MANIFEST)
ref_ia = pd.read_csv(CAMINHO_REF_IA)

respostas = respostas.merge(manifest[["item_id", "modelo", "prompt_variant"]], on="item_id", how="left")
print(f"Respostas professores: {len(respostas)} linhas")
print(f"Avaliadores únicos: {respostas['avaliador_id'].nunique()}")
respostas.head()

In [ ]:
def professores_long(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        for dim in DIMENSOES_TABELA1:
            col = COL_PROF[dim]
            if col not in row or pd.isna(row[col]):
                continue
            rows.append(
                {
                    "item_id": row["item_id"],
                    "modelo": row.get("modelo"),
                    "prompt": row.get("prompt_variant"),
                    "avaliador": row["avaliador_id"],
                    "grupo": "professor",
                    "dimensao": dim,
                    "likert": int(row[col]),
                }
            )
    return pd.DataFrame(rows)


def ia_long(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        juiz = ROTULO_JUIZ.get(row["juiz"], row["juiz"])
        for dim in DIMENSOES_TABELA1:
            col = f"likert_{dim}"
            if col not in row or pd.isna(row[col]):
                continue
            rows.append(
                {
                    "item_id": row["item_id"],
                    "modelo": row["modelo"],
                    "prompt": row["prompt_variant"],
                    "avaliador": juiz,
                    "grupo": "ia",
                    "dimensao": dim,
                    "likert": int(row[col]),
                }
            )
    return pd.DataFrame(rows)


item_ids = set(respostas["item_id"].unique())
ref_subset = ref_ia[ref_ia["item_id"].isin(item_ids)]

long_prof = professores_long(respostas)
long_ia = ia_long(ref_subset)
long_all = pd.concat([long_prof, long_ia], ignore_index=True)
print(long_all.groupby(["grupo", "avaliador"]).size())

In [ ]:
def formato_m_dp(media, dp):
    return f"{media:.2f} ({dp:.2f})"


stats = (
    long_all.groupby(["grupo", "avaliador", "dimensao"], as_index=False)["likert"]
    .agg(media="mean", dp="std", n="count")
)
stats["M_DP"] = stats.apply(lambda r: formato_m_dp(r["media"], r["dp"]), axis=1)
stats["dimensao_label"] = stats["dimensao"].map(ROTULOS_DIMENSAO)

tabela = stats.pivot_table(
    index=["grupo", "avaliador"],
    columns="dimensao_label",
    values="M_DP",
    aggfunc="first",
)
display(tabela)

In [ ]:
def comparar_professor_vs_juiz(juiz_rotulo: str):
    prof = long_prof
    ia = long_ia[long_ia["avaliador"] == juiz_rotulo]
    merged = prof.merge(
        ia,
        on=["item_id", "dimensao"],
        suffixes=("_prof", "_ia"),
    )
    rows = []
    for dim in DIMENSOES_TABELA1:
        sub = merged[merged["dimensao"] == dim]
        if len(sub) < 3:
            continue
        try:
            _, p = wilcoxon(sub["likert_prof"], sub["likert_ia"])
        except ValueError:
            p = np.nan
        rows.append(
            {
                "juiz": juiz_rotulo,
                "dimensao": ROTULOS_DIMENSAO[dim],
                "n_pares": len(sub),
                "media_prof": sub["likert_prof"].mean(),
                "media_ia": sub["likert_ia"].mean(),
                "diff_abs": abs(sub["likert_prof"].mean() - sub["likert_ia"].mean()),
                "p_wilcoxon": p,
            }
        )
    return pd.DataFrame(rows)


comps = pd.concat(
    [comparar_professor_vs_juiz(j) for j in ROTULO_JUIZ.values()],
    ignore_index=True,
)
display(comps.sort_values(["juiz", "dimensao"]))

In [ ]:
pivot = (
    long_all.groupby(["avaliador", "dimensao"])["likert"]
    .mean()
    .unstack("dimensao")[DIMENSOES_TABELA1]
)
pivot.columns = [ROTULOS_DIMENSAO[c] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(10, max(3, 0.45 * len(pivot))))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    vmin=1,
    vmax=3,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Média Likert 1–3 por avaliador e dimensão de conteúdo")
ax.set_xlabel("Dimensão")
plt.tight_layout()
plt.show()

In [ ]:
def extrair_justificativas_conteudo(valor, dim: str):
    try:
        dados = json.loads(valor) if isinstance(valor, str) else valor
        info = dados.get("conteudo", {}).get(dim)
        if info is None and dim == "pontos_fracos":
            info = dados.get("conteudo", {}).get("pontos_fraços")
        if not info:
            return None
        return info.get("justificativa")
    except (json.JSONDecodeError, TypeError, AttributeError):
        return None


if CAMINHO_CONSOLIDADO.exists():
    raw = pd.read_csv(CAMINHO_CONSOLIDADO, usecols=["pergunta", "resposta_aluno", "feedback", "modelo", "prompt_variant", "juiz", "analise_feedback"])
    manifest_items = manifest[["item_id", "modelo", "prompt_variant", "pergunta", "resposta_aluno", "feedback"]]
    detalhe_rows = []
    for _, m in manifest_items.iterrows():
        mask = (
            (raw["modelo"] == m["modelo"])
            & (raw["prompt_variant"] == m["prompt_variant"])
            & (raw["pergunta"] == m["pergunta"])
            & (raw["resposta_aluno"] == m["resposta_aluno"])
            & (raw["feedback"] == m["feedback"])
        )
        sub = raw.loc[mask]
        for _, r in sub.iterrows():
            for dim in DIMENSOES_TABELA1:
                txt = extrair_justificativas_conteudo(r["analise_feedback"], dim)
                if txt:
                    detalhe_rows.append(
                        {
                            "item_id": m["item_id"],
                            "juiz": ROTULO_JUIZ.get(r["juiz"], r["juiz"]),
                            "dimensao": dim,
                            "n_chars_justificativa_ia": len(txt),
                        }
                    )
    df_det_ia = pd.DataFrame(detalhe_rows)
    prof_coment = respostas[["item_id", "avaliador_id", "comentario_opcional"]].copy()
    prof_coment["n_chars_comentario_prof"] = prof_coment["comentario_opcional"].fillna("").str.len()
    print("Comprimento médio comentário professor:", prof_coment["n_chars_comentario_prof"].mean())
    print("Comprimento médio justificativa IA (conteúdo, por dimensão):")
    display(df_det_ia.groupby(["juiz", "dimensao"])["n_chars_justificativa_ia"].mean().unstack())
else:
    print("CSV consolidado não encontrado; pulando análise de detalhe textual.")